# Underwater Waste Detection — Evaluation

This notebook evaluates the trained YOLOv8 model on the test set and produces:
- mAP@0.5 and mAP@0.5:0.95 per class
- Precision / Recall curves
- Confusion matrix
- Sample prediction visualizations
- Inference speed benchmark (FPS)
- Comparison table (YOLOv8n vs YOLOv8m if both trained)

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
!pip install -q ultralytics seaborn

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE       = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DATASET_DIR = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR      = DRIVE_BASE / 'weights'
RESULTS_DIR      = DRIVE_BASE / 'evaluation_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Clone repo (or pull if exists)
REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
os.chdir(REPO_DIR)

import yaml

# Re-generate dataset YAML with correct absolute path
dataset_config = {
    'path': str(YOLO_DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {0: 'trash', 1: 'bio', 2: 'rov'}
}
yaml_path = REPO_DIR / 'configs' / 'dataset_colab.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f)

print('Setup complete.')
print(f'Looking for weights in: {WEIGHTS_DIR}')
for w in sorted(WEIGHTS_DIR.glob('*.pt')):
    print(f'  {w.name}  ({w.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# ── Cell 2: Run validation on val/test split ───────────────────────────────
from ultralytics import YOLO
import json

NANO_WEIGHTS = WEIGHTS_DIR / 'yolov8n_best.pt'
EVAL_SPLIT = 'val'  # change to 'test' if test split exists

print(f'Evaluating: {NANO_WEIGHTS.name}')
detector = YOLO(str(NANO_WEIGHTS))

val_results = detector.val(
    data=str(yaml_path),
    split=EVAL_SPLIT,
    conf=0.25,
    iou=0.45,
    verbose=True,
)

# Extract and save metrics
metrics = {
    'model': 'YOLOv8n',
    'split': EVAL_SPLIT,
    'mAP50': round(float(val_results.box.map50), 4),
    'mAP50_95': round(float(val_results.box.map), 4),
    'precision': round(float(val_results.box.mp), 4),
    'recall': round(float(val_results.box.mr), 4),
    'per_class_mAP50': [round(float(v), 4) for v in val_results.box.ap50.tolist()],
    'class_names': ['trash', 'bio', 'rov'],
}

metrics_path = RESULTS_DIR / 'yolov8n_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n' + '='*50)
print('EVALUATION RESULTS — YOLOv8n')
print('='*50)
print(f"  mAP@0.5:       {metrics['mAP50']:.4f}")
print(f"  mAP@0.5:0.95:  {metrics['mAP50_95']:.4f}")
print(f"  Precision:     {metrics['precision']:.4f}")
print(f"  Recall:        {metrics['recall']:.4f}")
print(f'\nPer-class mAP@0.5:')
for cls, ap in zip(metrics['class_names'], metrics['per_class_mAP50']):
    print(f"  {cls:<10} {ap:.4f}")
print('='*50)

In [ ]:
# ── Cell 3: Precision-Recall curves ───────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

CLASS_NAMES = ['trash', 'bio', 'rov']
CLASS_COLORS = ['#e74c3c', '#27ae60', '#3498db']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mAP per class bar chart
per_class_mAP = metrics['per_class_mAP50']
bars = axes[0].bar(CLASS_NAMES, per_class_mAP, color=CLASS_COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Per-class mAP@0.5 — YOLOv8n', fontweight='bold')
axes[0].set_ylabel('mAP@0.5')
axes[0].set_ylim(0, 1.0)
axes[0].axhline(y=metrics['mAP50'], color='gray', linestyle='--', label=f"Mean: {metrics['mAP50']:.3f}")
axes[0].legend()
for bar, val in zip(bars, per_class_mAP):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

# Summary metrics radar-style bar
summary_metrics = {
    'mAP@0.5': metrics['mAP50'],
    'mAP@0.5:0.95': metrics['mAP50_95'],
    'Precision': metrics['precision'],
    'Recall': metrics['recall'],
}
bars2 = axes[1].bar(summary_metrics.keys(), summary_metrics.values(),
                    color=['#2980b9', '#8e44ad', '#e67e22', '#16a085'], edgecolor='white')
axes[1].set_title('Summary Metrics — YOLOv8n', fontweight='bold')
axes[1].set_ylim(0, 1.0)
for bar, (k, v) in zip(bars2, summary_metrics.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('YOLOv8n Baseline — Underwater Waste Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'metrics_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 4: Visualize sample predictions ─────────────────────────────────
import cv2
from PIL import Image
import random

CLASS_COLORS_BGR = {'trash': (0, 0, 220), 'bio': (0, 180, 0), 'rov': (200, 130, 0)}

test_images = list((YOLO_DATASET_DIR / 'images' / EVAL_SPLIT).glob('*.jpg'))
test_images += list((YOLO_DATASET_DIR / 'images' / EVAL_SPLIT).glob('*.png'))
sampled = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, img_path in enumerate(sampled):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue

    preds = detector(frame, conf=0.25, iou=0.45, verbose=False)[0]
    annotated = frame.copy()

    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_idx = int(box.cls[0].item())
        conf_score = float(box.conf[0].item())
        label = CLASS_NAMES[cls_idx] if cls_idx < 3 else 'unknown'
        color = CLASS_COLORS_BGR.get(label, (128, 128, 128))

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        text = f"{label} {conf_score:.0%}"
        cv2.putText(annotated, text, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[idx].axis('off')
    n_trash = sum(1 for b in preds.boxes if int(b.cls[0]) == 0)
    axes[idx].set_title(f'{len(preds.boxes)} detections ({n_trash} trash)', fontsize=9)

plt.suptitle('YOLOv8n Sample Predictions — Underwater Waste Detection',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/sample_predictions.png')

In [ ]:
# ── Cell 5: Inference speed benchmark ─────────────────────────────────────
import time

bench_images = test_images[:50]
frames = [cv2.imread(str(p)) for p in bench_images if cv2.imread(str(p)) is not None]

# Warmup
for frame in frames[:5]:
    _ = detector(frame, verbose=False)

# Benchmark
start_time = time.time()
for frame in frames:
    _ = detector(frame, verbose=False)
elapsed = time.time() - start_time

fps = len(frames) / elapsed
ms_per_frame = (elapsed / len(frames)) * 1000

print(f'Inference speed (YOLOv8n):')
print(f'  {fps:.1f} FPS')
print(f'  {ms_per_frame:.1f} ms / frame')
print(f'  Tested on {len(frames)} images')

In [ ]:
# ── Cell 6: Results comparison table (update after training YOLOv8m) ──────
import pandas as pd

# Results table — update with actual numbers after training each model
results_table = pd.DataFrame([
    {
        'Model': 'YOLOv8n',
        'Params (M)': 3.2,
        'GFLOPs': 8.7,
        'mAP@0.5': metrics['mAP50'],
        'mAP@0.5:0.95': metrics['mAP50_95'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'FPS': round(fps, 1),
        'Status': 'trained'
    },
    {
        'Model': 'YOLOv8m',
        'Params (M)': 25.9,
        'GFLOPs': 78.9,
        'mAP@0.5': '—',
        'mAP@0.5:0.95': '—',
        'Precision': '—',
        'Recall': '—',
        'FPS': '—',
        'Status': 'pending'
    },
])

print('Model Comparison Table')
print('='*80)
print(results_table.to_string(index=False))

results_table.to_csv(str(RESULTS_DIR / 'model_comparison.csv'), index=False)
print(f'\nSaved: {RESULTS_DIR}/model_comparison.csv')

In [ ]:
# ── Cell 7: Export to ONNX (optional — for deployment) ───────────────────
# Uncomment to export the trained model to ONNX format

# onnx_path = WEIGHTS_DIR / 'yolov8n_underwater.onnx'
# detector.export(format='onnx', imgsz=640, dynamic=False)
# print(f'ONNX model exported.')

## Next Step: Improvement

The baseline is set. Now choose one improvement to implement in `04_improvement.ipynb`:

| Option | Description | Expected Gain |
|--------|-------------|---------------|
| **A. Underwater Preprocessing** | CLAHE + white balance before detection | +2–5% mAP on turbid/dark images |
| **B. Domain Augmentation** | Synthetic haze, color cast, caustic patterns during training | +3–6% mAP on varied conditions |
| **C. Small Object Handling** | SAHI tiling inference or higher-res training | +5–10% on small trash fragments |
| **D. Class Imbalance** | Focal loss + balanced sampling | Improved recall for rare trash classes |

Run the next cell to indicate your choice.

In [ ]:
# ── Cell 8: Choose improvement ────────────────────────────────────────────
# Set this to 'A', 'B', 'C', or 'D'
IMPROVEMENT_CHOICE = 'A'

descriptions = {
    'A': 'Underwater image preprocessing (CLAHE + white balance)',
    'B': 'Domain-specific data augmentation (haze, color cast, caustics)',
    'C': 'Small object detection (SAHI tiling inference)',
    'D': 'Class imbalance handling (focal loss + balanced sampling)',
}

print(f'Selected improvement: ({IMPROVEMENT_CHOICE}) {descriptions[IMPROVEMENT_CHOICE]}')
print('\nOpen 04_improvement.ipynb to implement this.')